In [1]:
import os
import sys
import glob
import pandas as pd

#avoid import errors
parent_dir=os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from scripts.CDS_machine import CDS_2_gffcomp
import scripts.get_busco_db as bob
import scripts.get_genetic_code as ggc
from scripts.counting_machine import write_counts, CATEGORIES


In [2]:
raw_data="../../data/species"
query_email="wsxktjnfwcecxnotrf@gonrr.net"
busco_database="/no_backup/rg/references/busco_downloads"

!ls $raw_data

Acanthamoeba_polyphaga_5757		   Micromonas_pusilla_CCMP1545_38833
Babesia_duncani_323732			   Naegleria_gruberi_5762
Caulerpa_lentillifera_148947		   Paramecium_bursaria_74790
Chaetoceros_neogracilis_240364		   Paramecium_tetraurelia_5888
Chlamydomonas_reinhardtii_3055		   Phytophthora_agathidicida_1642459
Chlorella_sorokiniana_3076		   Plasmodium_berghei_ANKA_5821
Cladocopium_goreaui_2562237		   Plasmodium_falciparum_3D7_36329
Conticribra_weissflogii_1577725		   Plasmodium_ovale_36330
Cryptosporidium_parvum_Iowa_II_5807	   Plasmodium_vivax_5855
Cyanidiococcus_yangmingshanensis_2690220   Plasmodium_yoelii_yoelii_73239
Cyanidioschyzon_merolae_strain_10D_280699  Pseudo-nitzschia_delicatissima_44447
Cyanidium_caldarium_2771		   Pseudo-nitzschia_multiseries_37319
Cylindrotheca_closterium_2856		   Pseudo-nitzschia_pungens_37318
Dunaliella_salina_3046			   Pyrocystis_lunula_2972
Eimeria_necatrix_51315			   Pyropia_haitanensis_1262161
Entamoeba_histolytica_HM-1:IMSS_5759	   Pyropia_yezoensis_

# Merge reference and predicted annotations

Use agat, as there are no in-python tools that do all the conflict resolution

In [3]:
#merge each prediction with the reference CDS of its own source
pred_units=[os.path.basename(p)[:-len(".gff3")] for p in sorted(glob.glob("../results/pred/*.gff3"))]
print(len(pred_units))

#checker(if things in file)
fileWritten=False
filepath="../job_commands/mergeAnn.txt"
with open(filepath, "w") as f:
    print("mkdir -p ../results/merged", file=f)
    for unit in pred_units:
        try:
            sp=unit.rsplit("_", 1)[0]
            pred=os.path.abspath(f"../results/pred/{unit}.gff3")

            #reference annotation must match the unit's source
            RefAnn=os.path.abspath(f"../training_data/species/{sp}/CDS_{unit}.gff")
            if not os.path.isfile(RefAnn):
                raise FileNotFoundError(f"There is no reference CDS for {unit}")

            fileWritten=True

            #create folder to store outputs
            merged_res=f"../results/merged/{unit}.gff"

            #write agat commands
            #per-task AGAT config so parallel array tasks don't collide on agat_config.yaml
            agat_cfg=f"agat_{unit}.yaml"
            expose_cmd=f"agat config --expose --output {agat_cfg} >/dev/null 2>&1"
            print(expose_cmd); print(expose_cmd, file=f)
            agat_merge=f"agat_sp_merge_annotations.pl --gff {RefAnn} --gff {pred} --config {agat_cfg} --out {merged_res}"

            print(agat_merge)
            print(agat_merge, file=f)
            clean_cmd=f"rm -f {agat_cfg}"
            print(clean_cmd); print(clean_cmd, file=f)

        except Exception as e:
            print(f"{unit} presents error: {e}")
            continue
        
if not fileWritten: #if the file is not written, delete it
    os.remove(filepath)
else: #copy the matching array submitter next to its command file
    !cp ../scripts/memory/merger.sh ../job_commands/
            

65
agat config --expose --output agat_Babesia_duncani_323732_lyric.yaml >/dev/null 2>&1
agat_sp_merge_annotations.pl --gff /users/rg/jizquierdo/git/geneid-training/training_data/species/Babesia_duncani_323732/CDS_Babesia_duncani_323732_lyric.gff --gff /users/rg/jizquierdo/git/geneid-training/results/pred/Babesia_duncani_323732_lyric.gff3 --config agat_Babesia_duncani_323732_lyric.yaml --out ../results/merged/Babesia_duncani_323732_lyric.gff
rm -f agat_Babesia_duncani_323732_lyric.yaml
agat config --expose --output agat_Babesia_duncani_323732_reference.yaml >/dev/null 2>&1
agat_sp_merge_annotations.pl --gff /users/rg/jizquierdo/git/geneid-training/training_data/species/Babesia_duncani_323732/CDS_Babesia_duncani_323732_reference.gff --gff /users/rg/jizquierdo/git/geneid-training/results/pred/Babesia_duncani_323732_reference.gff3 --config agat_Babesia_duncani_323732_reference.yaml --out ../results/merged/Babesia_duncani_323732_reference.gff
rm -f agat_Babesia_duncani_323732_reference.yaml

In [3]:
mg=!ls ../results/merged
print(len(mg))
print(mg)

65
['Babesia_duncani_323732_lyric.gff', 'Babesia_duncani_323732_reference.gff', 'Chaetoceros_neogracilis_240364_lyric.gff', 'Chaetoceros_neogracilis_240364_reference.gff', 'Chlamydomonas_reinhardtii_3055_isoquant.gff', 'Chlamydomonas_reinhardtii_3055_lyric.gff', 'Chlamydomonas_reinhardtii_3055_reference.gff', 'Chlorella_sorokiniana_3076_lyric.gff', 'Conticribra_weissflogii_1577725_lyric.gff', 'Conticribra_weissflogii_1577725_reference.gff', 'Cryptosporidium_parvum_Iowa_II_353152_reference.gff', 'Cryptosporidium_parvum_Iowa_II_5807_lyric.gff', 'Cyanidiococcus_yangmingshanensis_2690220_lyric.gff', 'Cyanidiococcus_yangmingshanensis_2690220_reference.gff', 'Cyanidioschyzon_merolae_strain_10D_280699_lyric.gff', 'Cyanidioschyzon_merolae_strain_10D_280699_reference.gff', 'Cyanidium_caldarium_2771_lyric.gff', 'Cyanidium_caldarium_2771_reference.gff', 'Cylindrotheca_closterium_2856_lyric.gff', 'Cylindrotheca_closterium_2856_reference.gff', 'Dunaliella_salina_3046_lyric.gff', 'Dunaliella_salina_

# Prediction analysis

## Reference annotation(CDS) and Prediction comparison

## BUSCO assessment

In [8]:
#agat for gff>prot
#evaluate every merged annotation: <species>_<source>.gff
merged_units=[os.path.basename(p)[:-len(".gff")] for p in sorted(glob.glob("../results/merged/*.gff"))]
print(len(merged_units))

#checker(if things in file)
fileWritten=False
filepath="../job_commands/M_buscoScoring.txt"
with open(filepath, "w") as f:
    print('cpus="${SLURM_CPUS_PER_TASK:-$(nproc)}"', file=f)
    print(f"mkdir -p ../results/summary/merged/busco_lineage ../results/summary/merged/busco_eukaryote", file=f)
    for unit in merged_units:
        try:
            sp=unit.rsplit("_", 1)[0]
            merged=os.path.abspath(f"../results/merged/{unit}.gff")

            fileWritten=True

            #reference sequence (shared genome for the species)
            RefSeq=!realpath ../training_data/species/$sp/CLEAN_*.fna
            RefSeq=RefSeq[0]

            #create folder to store outputs
            prot_res=f"../results/specie_logs/{unit}/{unit}_merged_prot.fa" #merged protein sequence output
            busco_outPath=f"../results/merged_busco/" #busco comparisons output
            Lbusco_res=f"{unit}_Lbusco" #taxon-lineage busco out name
            Ebusco_res=f"{unit}_Ebusco" #eukaryota busco out name

            #get species taxon
            taxID=sp[sp.rfind("_")+1:]
            #find lineage
            busco_lineage=bob.taxon2lineage(taxID, query_email, f"{busco_database}/file_versions.tsv", "odb12")
            euk_lineage="eukaryota_odb12"
            #resolve the NCBI nuclear genetic code for this taxon (table 1 if unresolved)
            gcode=ggc.taxon2geneticcode(taxID, query_email) or 1

            #gff to protein command
            #per-task AGAT config so parallel array tasks don't collide on agat_config.yaml
            agat_cfg=f"agat_{unit}_merged.yaml"
            expose_cmd=f"agat config --expose --output {agat_cfg} >/dev/null 2>&1"
            print(expose_cmd); print(expose_cmd, file=f)
            TOprotein_cmd=f"agat_sp_extract_sequences.pl -g {merged} -f {RefSeq} -t cds -p --table {gcode} --config {agat_cfg} -o {prot_res}"

            print(TOprotein_cmd)
            print(TOprotein_cmd, file=f)
            clean_cmd=f"rm -f {agat_cfg}"
            print(clean_cmd); print(clean_cmd, file=f)
            #deduplicate protein IDs (AGAT can emit duplicate names that break BUSCO)
            ND_prot_res=prot_res.replace(".fa", "_ND.fa")
            rename_cmd=f"seqkit rename -n {prot_res} > {ND_prot_res}"
            print(rename_cmd)
            print(rename_cmd, file=f)
            
            Lbusco_cmd=f"busco -m protein \
                        -i {ND_prot_res} \
                        --download_path {busco_database} \
                        -l {busco_lineage} \
                        --cpu $cpus \
                        -f \
                        --opt-out-run-stats \
                        --out_path {busco_outPath} \
                        -o {Lbusco_res}" #--offline #autolineage creates errors
            Lbusco_cmd=Lbusco_cmd.replace("                             ", " ")

            Ebusco_cmd=f"busco -m protein \
                        -i {ND_prot_res} \
                        --download_path {busco_database} \
                        -l {euk_lineage} \
                        --cpu $cpus \
                        -f \
                        --opt-out-run-stats \
                        --out_path {busco_outPath} \
                        -o {Ebusco_res}" #--offline #autolineage creates errors
            Ebusco_cmd=Ebusco_cmd.replace("                             ", " ")

            print(Lbusco_cmd); print(Lbusco_cmd, file=f)
            print(Ebusco_cmd); print(Ebusco_cmd, file=f)

            #collect each run JSON into its own summary folder (isoquant naming)
            print(f"ln -vf {busco_outPath}{Lbusco_res}/short_summary.specific.*.json ../results/summary/merged/busco_lineage/{Lbusco_res}.json", file=f)
            print(f"ln -vf {busco_outPath}{Ebusco_res}/short_summary.specific.*.json ../results/summary/merged/busco_eukaryote/{Ebusco_res}.json", file=f)

        except Exception as e:
            print(f"{unit} presents error: {e}")
            continue

        
if not fileWritten: #if the file is not written, delete it
    os.remove(filepath)
else: #copy the matching array submitter next to its command file
    !cp ../scripts/memory/busco.sh ../job_commands/
            

65
agat config --expose --output agat_Babesia_duncani_323732_lyric_merged.yaml >/dev/null 2>&1
agat_sp_extract_sequences.pl -g /users/rg/jizquierdo/git/geneid-training/results/merged/Babesia_duncani_323732_lyric.gff -f /users/rg/jizquierdo/git/geneid-training/training_data/species/Babesia_duncani_323732/CLEAN_GCF_028658345.1_Bduncani_v1_genomic.fna -t cds -p --table 1 --config agat_Babesia_duncani_323732_lyric_merged.yaml -o ../results/specie_logs/Babesia_duncani_323732_lyric/Babesia_duncani_323732_lyric_merged_prot.fa
rm -f agat_Babesia_duncani_323732_lyric_merged.yaml
seqkit rename -n ../results/specie_logs/Babesia_duncani_323732_lyric/Babesia_duncani_323732_lyric_merged_prot.fa > ../results/specie_logs/Babesia_duncani_323732_lyric/Babesia_duncani_323732_lyric_merged_prot_ND.fa
busco -m protein                         -i ../results/specie_logs/Babesia_duncani_323732_lyric/Babesia_duncani_323732_lyric_merged_prot_ND.fa                         --download_path /no_backup/rg/references/b

In [4]:
!ls ../results/summary/merged/busco_eukaryote

Babesia_duncani_323732_lyric_Ebusco.json
Babesia_duncani_323732_reference_Ebusco.json
Chaetoceros_neogracilis_240364_lyric_Ebusco.json
Chaetoceros_neogracilis_240364_reference_Ebusco.json
Chlamydomonas_reinhardtii_3055_isoquant_Ebusco.json
Chlamydomonas_reinhardtii_3055_lyric_Ebusco.json
Chlamydomonas_reinhardtii_3055_reference_Ebusco.json
Chlorella_sorokiniana_3076_lyric_Ebusco.json
Conticribra_weissflogii_1577725_lyric_Ebusco.json
Conticribra_weissflogii_1577725_reference_Ebusco.json
Cryptosporidium_parvum_Iowa_II_353152_reference_Ebusco.json
Cryptosporidium_parvum_Iowa_II_5807_lyric_Ebusco.json
Cyanidiococcus_yangmingshanensis_2690220_lyric_Ebusco.json
Cyanidiococcus_yangmingshanensis_2690220_reference_Ebusco.json
Cyanidioschyzon_merolae_strain_10D_280699_lyric_Ebusco.json
Cyanidioschyzon_merolae_strain_10D_280699_reference_Ebusco.json
Cyanidium_caldarium_2771_lyric_Ebusco.json
Cyanidium_caldarium_2771_reference_Ebusco.json
Cylindrotheca_closterium_2856_lyric_Ebusco.json
Cylindrothe

# Summary

The **regular** evaluation outputs are collected under `results/summary/regular/`: the busco cells above write the BUSCO summaries

In [5]:
#metrics for the regular predictions -> results/summary/regular/counts/ per-species file + counts_summary.tsv.

results_dir = "../results"
summary_dir = f"{results_dir}/summary"
os.makedirs(summary_dir, exist_ok=True)

write_counts(results_dir, summary_dir, "merged", CATEGORIES["merged"])

[merged] metrics: 46 species (65 units) -> counts/ (+ counts_summary.tsv)


([['Babesia_duncani_323732',
   '4453',
   '9658',
   '10408054',
   'NA',
   'NA',
   '427.84',
   '927.94',
   '2.169',
   'NA'],
  ['Chaetoceros_neogracilis_240364',
   '15100',
   '42836',
   '38617829',
   'NA',
   'NA',
   '391.01',
   '1109.23',
   '2.837',
   'NA'],
  ['Chlamydomonas_reinhardtii_3055',
   '24522',
   '66781',
   '111098438',
   'NA',
   'NA',
   '220.72',
   '601.10',
   '2.723',
   'NA'],
  ['Chlorella_sorokiniana_3076',
   '10866',
   '26056',
   '38807484',
   'NA',
   'NA',
   '280.00',
   '671.42',
   '2.398',
   'NA'],
  ['Conticribra_weissflogii_1577725',
   '49429',
   '65899',
   '129999045',
   'NA',
   'NA',
   '380.23',
   '506.92',
   '1.333',
   'NA'],
  ['Cryptosporidium_parvum_Iowa_II_353152',
   '3878',
   '4691',
   '9102324',
   'NA',
   'NA',
   '426.05',
   '515.36',
   '1.210',
   'NA'],
  ['Cryptosporidium_parvum_Iowa_II_5807',
   '3833',
   '8090',
   '9102324',
   'NA',
   'NA',
   '421.10',
   '888.78',
   '2.111',
   'NA'],
  ['Cyanid